# Audio Hybrid Neuroevolution - Refactored

This notebook demonstrates the modular neuroevolution package in action.

## Features

- **Modular Architecture**: All logic extracted into reusable Python modules
- **Hybrid Neuroevolution**: Genetic algorithms + supervised training
- **Parallel 5-fold validation protocol**: All folds evaluated simultaneously using ThreadPoolExecutor
- **NEAT-like Innovation Tracking**: Gene alignment for crossover
- **Genetic Speciation**: Compatibility-based species assignment
- **Incremental Complexity Growth**: Networks start simple, grow gradually
- **Adaptive Mutation**: Rates adjust based on population diversity
- **Complete Metrics**: Accuracy, Sensitivity, Specificity, Precision, F1, AUC

## Workflow

1. Setup environment and configuration
2. Load and verify data
3. Initialize neuroevolution engine
4. Execute evolution process
5. Visualize results and analyze best architecture


## 1. Environment Setup and Imports

In [1]:
# Install required packages if needed
import os
from neuroevolution import install_packages, setup_logging

ARTIFACTS_DIR = os.path.join('artifacts', 'test_audio')
setup_logging(os.path.join(ARTIFACTS_DIR, 'execution_log.txt'), echo_to_console=False)

packages = [
    "torch==2.11.0",
    "numpy>=1.21.0",
    "matplotlib>=3.5.0",
    "seaborn>=0.11.0",
    "scikit-learn>=1.0.0"
]

install_packages(packages)

OSError: [WinError 4551] Una directiva de Control de aplicaciones bloqueó este archivo. Error loading "C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [2]:
# Core imports
import os
from datetime import datetime

# Neuroevolution package imports
from neuroevolution import (
    # Configuration and setup
    CONFIG,
    setup_device,
    setup_seeds,
    setup_logging,
    # Data loading
    load_dataset,
    # Evolution engine
    HybridNeuroevolution,
    # Final held-out evaluation and reporting
    evaluate_5fold_cross_validation,
    format_held_out_results_markdown,
    plot_fold_confusion_matrices,
    # Visualization
    plot_fitness_evolution,
    show_evolution_statistics,
    analyze_failed_evaluations,
    configure_plot_style
)
from neuroevolution.data.loader import load_fold_data as load_fold_arrays

print("✅ All modules imported successfully")

OSError: [WinError 4551] Una directiva de Control de aplicaciones bloqueó este archivo. Error loading "C:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 2. Configuration and Device Setup

In [3]:
# Configure data paths
CONFIG['data_path'] = os.path.join('data', 'sets', 'folds_5')
CONFIG['artifacts_dir'] = os.path.join('artifacts', 'test_audio')

# Optional: Adjust parameters for quick testing
# CONFIG['population_size'] = 8
# CONFIG['max_generations'] = 20
# CONFIG['fitness_threshold'] = 85.0

# Setup logging (redirects print to file)
log_path = os.path.join(CONFIG['artifacts_dir'], 'execution_log.txt')
setup_logging(log_path, echo_to_console=False)

# Setup device and seeds for reproducibility
device = setup_device()
setup_seeds(42)

# Configure plot style
configure_plot_style()

print("="*60)
print("AUDIO NEUROEVOLUTION CONFIGURATION")
print("="*60)
print(f"Dataset: Audio (Parkinson Classification)")
print(f"Dataset ID: {CONFIG['dataset_id']}")
print(f"Fold ID: {CONFIG['fold_id']}")
print(f"Data Path: {CONFIG['data_path']}")
print(f"Artifacts Path: {CONFIG['artifacts_dir']}")
print(f"Population: {CONFIG['population_size']} individuals")
print(f"Maximum generations: {CONFIG['max_generations']}")
print(f"Target fitness: {CONFIG['fitness_threshold']}%")
print(f"Device: {device}")
print(f"Parallelization: Enabled (5 threads per individual)")
print("="*60)

NameError: name 'CONFIG' is not defined

## 3. Data Loading and Verification

In [4]:
print("Verifying audio dataset...\n")

# Verify all folds and auto-detect sequence length
load_dataset(CONFIG)

# Load fold 1 only for quick sample-count reporting
X_train, y_train, X_val, y_val, X_test, y_test = load_fold_arrays(CONFIG, fold_num=1)

print(f"\n{'='*60}")
print("DATASET VERIFIED - READY FOR PARALLEL 5-FOLD VALIDATION EVOLUTION")
print(f"{'='*60}")
print(f"Sequence length detected: {CONFIG['sequence_length']}")
print(f"Number of channels: {CONFIG['num_channels']}")
print(f"Number of classes: {CONFIG['num_classes']}")
print(f"Train samples (fold 1): {len(y_train)}")
print(f"Val samples (fold 1): {len(y_val)}")
print(f"Test samples (fold 1): {len(y_test)}")

Verifying audio dataset...



NameError: name 'load_dataset' is not defined

## 4. Evolution Execution

This cell runs the complete neuroevolution process:
- Initializes population with incremental complexity
- Evaluates each individual on all 5 folds IN PARALLEL
- Applies selection, crossover, and mutation
- Saves checkpoints and progress
- Continues until convergence or max generations

In [5]:
start_time = datetime.now()
print(f"\nStarting audio neuroevolution at {start_time.strftime('%H:%M:%S')}")
print(f"Architecture: Conv1D -> BatchNorm1D -> Activation -> MaxPool1D -> FC")
print(f"Each individual will be evaluated on all 5 folds IN PARALLEL")
print(f"Using ThreadPoolExecutor with 5 workers (one per fold)")
print(f"{'='*60}\n")

# Create neuroevolution instance
neuroevolution = HybridNeuroevolution(CONFIG, device)

# Execute evolution process
best_genome = neuroevolution.evolve()

end_time = datetime.now()
execution_time = end_time - start_time

print(f"\n{'='*60}")
print("EVOLUTION PROCESS COMPLETED")
print(f"{'='*60}")
print(f"Completed at: {end_time.strftime('%H:%M:%S')}")
print(f"Total execution time: {execution_time}")
print(f"Total generations: {neuroevolution.generation}")
print(f"Best fitness achieved: {best_genome['fitness']:.2f}%")
print(f"Progress JSON: {neuroevolution.progress_json_path}")
print(f"Generation log: {neuroevolution.generation_log_path}")
print(f"{'='*60}")


Starting audio neuroevolution at 10:56:17
Architecture: Conv1D -> BatchNorm1D -> Activation -> MaxPool1D -> FC
Each individual will be evaluated on all 5 folds IN PARALLEL
Using ThreadPoolExecutor with 5 workers (one per fold)



NameError: name 'HybridNeuroevolution' is not defined

## 5. Final Held-Out Test Evaluation

The evolutionary fitness above is selected from validation F1. This separate stage retrains the selected architecture per fold, uses validation for checkpoint selection, and reports final metrics only on held-out test data.

In [6]:
final_test_results = evaluate_5fold_cross_validation(
    best_genome=best_genome,
    config=CONFIG,
    device=device,
    neuroevolution_instance=neuroevolution,
)

if final_test_results is None:
    raise RuntimeError('Held-out test evaluation failed: no fold produced results. Check the execution log.')

print('Held-out test evaluation completed.')
print(f"Results JSON: {final_test_results['results_path']}")

NameError: name 'evaluate_5fold_cross_validation' is not defined

In [7]:
print('ARTICLE-READY HELD-OUT TEST RESULTS')
print(format_held_out_results_markdown(final_test_results))

confusion_matrix_figure = plot_fold_confusion_matrices(final_test_results)

ARTICLE-READY HELD-OUT TEST RESULTS


NameError: name 'format_held_out_results_markdown' is not defined

## 6. Results Visualization

In [8]:
# Plot fitness evolution across generations
plot_fitness_evolution(neuroevolution, CONFIG)

NameError: name 'plot_fitness_evolution' is not defined

In [9]:
# Show detailed statistics
show_evolution_statistics(neuroevolution, CONFIG)

NameError: name 'show_evolution_statistics' is not defined

In [10]:
# Analyze any failed evaluations
analyze_failed_evaluations(neuroevolution)

NameError: name 'analyze_failed_evaluations' is not defined

## 7. Best Architecture Analysis

Examine the best architecture found during evolution.

In [11]:
print("\n" + "="*80)
print("BEST ARCHITECTURE DETAILS")
print("="*80 + "\n")

best = best_genome

print(f"ID: {best['id']}")
print(f"Fitness (F1-Score): {best['fitness']:.2f}%\n")

print("ARCHITECTURE:")
print(f"  Convolutional layers: {best['num_conv_layers']}")
print(f"  Fully connected layers: {best['num_fc_layers']}")
print(f"\nCONV LAYER DETAILS:")
normalization = best.get('normalization_type', 'batch')
activations = best.get('activations', ['relu'])
for i in range(best['num_conv_layers']):
    activation = activations[i % len(activations)]
    print(f"  Conv{i+1}: {best['filters'][i]} filters, kernel={best['kernel_sizes'][i]}, norm={normalization}, activation={activation}")

print(f"\nFC LAYER DETAILS:")
for i in range(best['num_fc_layers']):
    print(f"  FC{i+1}: {best['fc_nodes'][i]} nodes")

print(f"\nHYPERPARAMETERS:")
print(f"  Optimizer: {best['optimizer']}")
print(f"  Learning rate: {best['learning_rate']}")
print(f"  Dropout rate: {best['dropout_rate']}")
print(f"  Activations: {', '.join(sorted(set(activations)))}")

if best.get('innovation_uuid'):
    print(f"\nINNOVATION TRACKING:")
    print(f"  Innovation UUID: {best['innovation_uuid'][:16]}...")
    print(f"  Number of innovation genes: {len(best.get('innovation_genes', []))}")
    if best.get('structural_history'):
        print(f"  Structural events: {len(best['structural_history'])}")

print("\n" + "="*80)


BEST ARCHITECTURE DETAILS



NameError: name 'best_genome' is not defined

## 8. Load Best Checkpoint (Optional)

You can reload the best model from checkpoint for inference or further analysis.

In [12]:
# Load best checkpoint
loaded_genome, loaded_model = neuroevolution.load_best_checkpoint()

if loaded_model is not None:
    print("\nBest model loaded successfully!")
    print(f"Model has {sum(p.numel() for p in loaded_model.parameters())} parameters")
    print(f"Trainable parameters: {sum(p.numel() for p in loaded_model.parameters() if p.requires_grad)}")
    print("\nModel architecture:")
    print(loaded_model)
else:
    print("No checkpoint available to load")

NameError: name 'neuroevolution' is not defined

## Summary

This notebook demonstrates the complete modular neuroevolution workflow:

1. ✅ **Modular imports** - All logic in reusable Python modules
2. ✅ **Configuration** - Centralized config with sensible defaults
3. ✅ **Data loading** - Automatic fold detection and validation
4. ✅ **Evolution** - Parallel 5-fold validation protocol with NEAT-like innovation tracking
5. ✅ **Held-out testing** - Independent test evaluation and article-ready results
6. ✅ **Visualization** - Comprehensive plots and statistics
7. ✅ **Analysis** - Detailed best architecture inspection

### Next Steps

- Experiment with different dataset configurations (`dataset_id`, `fold_id`)
- Adjust hyperparameters (population size, mutation rates, etc.)
- Compare results across multiple runs
- Compare held-out test results across multiple runs

### File Outputs

All results are saved in `artifacts/test_audio/`:
- `evolution_progress.json` - Complete evolution state (resumable)
- `generation_progress.txt` - Detailed metrics per generation
- `execution_log.txt` - Full execution log
- `best_model_*.pth` - Best model checkpoint with genome and config
- `5fold_cv_results_*.json` - Final held-out test metrics and per-fold confusion matrices